In [1]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [2]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Drosophila


In [3]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
!mkdir Droso_chemical_promotes_biologicalprocess
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Drosophila/Droso_chemical_promotes_biologicalprocess/Droso_chemical_promotes_biologicalprocess.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'species'
]

# AgingAtlas

In [4]:
AgingAtlas = pd.read_csv(PROC_DIR + 'agingatlas/Droso/Droso_AgingAtlas_Chemical_Promotes_BiologicalProcess_Geroprotectors.csv')
AgingAtlas.columns = AgingAtlas.columns.str.lower()
AgingAtlas = AgingAtlas[~AgingAtlas['head_detail_name'].isna()]

AgingAtlas['kg_type'] = 'Aging'
AgingAtlas['tail_type'] = 'BiologicalProcess'
print(f"AgingAtlas: {len(AgingAtlas):,} rows")
AgingAtlas['head_id_is'] = 'Pubchem'
AgingAtlas['tail_id_is'] = 'Quick_GO'
AgingAtlas['species'] =  'Drosophila melanogaster'
AgingAtlas['kg_source'] = 'AgingAtlas'
AgingAtlas

AgingAtlas: 2 rows


,head,relation,tail,head_type,tail_type,head_detail_name,head_smile_name,relation_source,species,head_alt_name,pmid,rnaseq,data_type,tail_detail_name,kg_type,head_id_is,tail_id_is,kg_source
0,6036,ChemicalEntity_Promotes_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,"(3R,4S,5R,6R)-6-(hydroxymethyl)oxane-2,3,4,5-t...",C([C@@H]1[C@@H]([C@@H]([C@H](C(O1)O)O)O)O)O,Aging_Atlas_gerprotector,Drosophila melanogaster,Galactose,15547319,Not Available,Causative,aging,Aging,Pubchem,Quick_GO,AgingAtlas
1,1123,ChemicalEntity_Promotes_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,2-aminoethanesulfonic acid,C(CS(=O)(=O)O)N,Aging_Atlas_gerprotector,Drosophila melanogaster,Taurine,29093334,Not Available,Causative,aging,Aging,Pubchem,Quick_GO,AgingAtlas


# Consolidate into Unified Schem

In [5]:
# List all source DataFrames to include
source_dfs = [
     AgingAtlas
    
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 2


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,6036,ChemicalEntity_Promotes_BiologicalProcess,GO:0007568,ChemicalEntity,None,BiologicalProcess,AgingAtlas,Aging,Pubchem,Quick_GO,"(3R,4S,5R,6R)-6-(hydroxymethyl)oxane-2,3,4,5-t...",aging,Drosophila melanogaster
1,1123,ChemicalEntity_Promotes_BiologicalProcess,GO:0007568,ChemicalEntity,None,BiologicalProcess,AgingAtlas,Aging,Pubchem,Quick_GO,2-aminoethanesulfonic acid,aging,Drosophila melanogaster


# Sanity Check — Distinct Values

In [6]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'ChemicalEntity_Promotes_BiologicalProcess'}
head_type           : {'ChemicalEntity'}
tail_type           : {'BiologicalProcess'}
relation_type       : {None}
kg_source           : {'AgingAtlas'}
head_id_is          : {'Pubchem'}
tail_id_is          : {'Quick_GO'}


In [7]:
# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 2 remaining


# NaN Audit (pre-dedup)

In [8]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,2,0,2
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


# Deduplication

In [9]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 2  |  After dedup: 2


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,1123,ChemicalEntity_Promotes_BiologicalProcess,GO:0007568,ChemicalEntity,None,BiologicalProcess,AgingAtlas,Aging,Pubchem,Quick_GO,2-aminoethanesulfonic acid,aging,Drosophila melanogaster
1,6036,ChemicalEntity_Promotes_BiologicalProcess,GO:0007568,ChemicalEntity,None,BiologicalProcess,AgingAtlas,Aging,Pubchem,Quick_GO,"(3R,4S,5R,6R)-6-(hydroxymethyl)oxane-2,3,4,5-t...",aging,Drosophila melanogaster


In [10]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,2,0,2
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [11]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'AgingAtlas'} kg_source
AgingAtlas    2
Name: count, dtype: int64


In [12]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'Aging'} kg_type
Aging    2
Name: count, dtype: int64


In [13]:
! mkdir Droso_chemical_promotes_biologicalprocess

mkdir: cannot create directory ‘Droso_chemical_promotes_biologicalprocess’: File exists


In [14]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 2 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Drosophila/Droso_chemical_promotes_biologicalprocess/Droso_chemical_promotes_biologicalprocess.csv


In [15]:
# OUT_PATH